In [1]:
import pandas as pd

customers = pd.read_csv("../data/olist_customers_dataset.csv")
orders    = pd.read_csv("../data/olist_orders_dataset.csv")
items     = pd.read_csv("../data/olist_order_items_dataset.csv")

print("Loaded")

Loaded


In [2]:
date_cols = ["order_purchase_timestamp", "order_delivered_customer_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
dtype: object

In [3]:
orders = orders[orders["order_status"] == "delivered"]
print(orders.shape)

(96478, 8)


In [4]:
order_value = items.groupby("order_id")["price"].sum().reset_index()
order_value.head()

,order_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90


In [5]:
df = orders.merge(order_value, on="order_id", how="left")
df = df.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")

print(df.shape)
df.head()

(96478, 10)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,72632f0f9dd73dfee390c9b22eb56dd6


In [6]:
print("Before dropping:", df.shape)
df = df.dropna(subset=["price", "order_purchase_timestamp"])
print("After dropping:", df.shape)

Before dropping: (96478, 10)
After dropping: (96478, 10)


In [7]:
df.to_csv("../data/clean_orders.csv", index=False)
print("Saved clean_orders.csv")

Saved clean_orders.csv


In [8]:
print("Total unique customers:", df["customer_unique_id"].nunique())
print("Total orders:", len(df))
print("Date range:", df["order_purchase_timestamp"].min(), "to", df["order_purchase_timestamp"].max())

Total unique customers: 93358
Total orders: 96478
Date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37


## Day 2 Notes
- Filtered to delivered orders only, merged order value from items table
- customer_unique_id used (not customer_id) since customer_id is per-order, not per-customer
- Saved clean_orders.csv — base table for all future analysis